In [ ]:
import gc
import os
import subprocess
import sys
import traceback


print("=" * 60)
print("NEMOTRON LORA TRAINING WITH SFT - v22 (Version 9)")
print("Knowledge Distillation from Teacher Model + Mamba-2/MoE Optimization")
print("=" * 60)

try:
    # 1. Blackwell Environment Setup
    print("\n[1/8] Setting up Blackwell environment...")
    UTILITY_PATH = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script"
    if os.path.exists(UTILITY_PATH):
        subprocess.run(f"tar -cf - -C {UTILITY_PATH} . | tar -xf - -C /tmp", shell=True, check=True)
        for binary in ["ptxas", "ptxas-blackwell"]:
            bin_path = f"/tmp/triton/backends/nvidia/bin/{binary}"
            if os.path.exists(bin_path):
                subprocess.run(f"chmod +x {bin_path}", shell=True, check=True)
        os.environ["TRITON_PTXAS_PATH"] = "/tmp/triton/backends/nvidia/bin/ptxas-blackwell"
        sys.path.insert(0, "/tmp")
        print("Blackwell environment initialized")
    else:
        print("WARNING: Utility script not found")

    # 2. Imports & Dependencies
    print("\n[2/8] Loading dependencies...")

    MANDATORY_PACKAGES = [
        "trl",
        "peft",
        "bitsandbytes",
        "accelerate",
        "nvidia-cutlass",
        "mamba_ssm",
        "causal_conv1d",
    ]
    print(f"Verifying mandatory packages: {MANDATORY_PACKAGES}")

    for pkg in MANDATORY_PACKAGES:
        try:
            __import__(pkg.replace("-", "_"))
            print(f"  {pkg} already installed")
        except ImportError:
            print(f"  Installing {pkg}...")
            # Use --no-build-isolation to avoid dependency loops or missing build tools in Kaggle environment
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", pkg]
            )

    import kagglehub
    import pandas as pd
    import torch
    from datasets import Dataset
    from peft import LoraConfig, get_peft_model
    from transformers import (
        AutoModelForCausalLM,
        AutoTokenizer,
        TrainingArguments,
    )
    from trl import SFTTrainer

    # 3. Hardware check
    print("\n[3/8] Hardware configuration:")
    print(f"  PyTorch: {torch.__version__}")
    print(f"  CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            prop = torch.cuda.get_device_properties(i)
            print(f"  GPU {i}: {prop.name} ({prop.total_memory / 1024**3:.1f} GB)")

    # 4. Load competition data
    print("\n[4/8] Loading training data...")
    competition_id = "nvidia-nemotron-model-reasoning-challenge"

    train_file = None
    for base_path in [f"/kaggle/input/{competition_id}", "/kaggle/input"]:
        if os.path.exists(base_path):
            for root, dirs, files in os.walk(base_path):
                for f in files:
                    if "train" in f.lower() and f.endswith(".csv"):
                        train_file = os.path.join(root, f)
                        break
                if train_file:
                    break
        if train_file:
            break

    if not train_file:
        print("ERROR: Training data not found")
        sys.exit(1)

    print(f"  Dataset: {train_file}")
    df = pd.read_csv(train_file)
    print(f"  Samples: {len(df)}")
    print(f"  Columns: {list(df.columns)}")

    # 5. Teacher trace generation (knowledge distillation)
    print("\n[5/8] Generating teacher traces for distillation...")

    # Load teacher model for generating CoT traces
    # Using DeepSeek-R1-Distill-Qwen-32B as teacher for reasoning traces
    teacher_model_name = "deepseek-ai/deepseek-r1-distill-qwen-32b"

    try:
        print(f"  Loading teacher: {teacher_model_name}")
        teacher_tokenizer = AutoTokenizer.from_pretrained(
            teacher_model_name, trust_remote_code=True
        )
        teacher_model = AutoModelForCausalLM.from_pretrained(
            teacher_model_name,
            torch_dtype=torch.bfloat16,
            device_map="auto",
            trust_remote_code=True,
        )
        print("  Teacher model loaded")

        # Generate reasoning traces for training samples
        def generate_teacher_trace(row):
            """Generate CoT reasoning trace from teacher model."""
            problem = row.get("problem", row.get("question", row.get("prompt")))
            prompt = f"""Solve this step by step and put your final answer in \\boxed{{}}.

Problem: {problem}

Let's think through this carefully:"""

            inputs = teacher_tokenizer(prompt, return_tensors="pt").to(teacher_model.device)

            with torch.no_grad():
                outputs = teacher_model.generate(
                    **inputs,
                    max_new_tokens=1024,
                    temperature=0.7,
                    do_sample=True,
                    pad_token_id=teacher_tokenizer.eos_token_id,
                )

            response = teacher_tokenizer.decode(outputs[0], skip_special_tokens=True)
            # Extract thinking and answer
            return response[len(prompt) :].strip()

        # Generate traces for first 100 samples (budget: ~$25)
        sample_size = min(100, len(df))
        print(f"  Generating traces for {sample_size} samples...")

        teacher_traces = []
        for idx in range(sample_size):
            row = df.iloc[idx]
            trace = generate_teacher_trace(row)
            teacher_traces.append(trace)
            if (idx + 1) % 10 == 0:
                print(f"    Generated {idx + 1}/{sample_size} traces")

        # Filter: keep only traces where teacher answer matches ground truth
        def extract_answer_from_trace(trace):
            """Extract integer answer from teacher trace."""
            import re

            # Look for \boxed{...}
            matches = re.findall(r"\\boxed\{([^}]+)\}", trace)
            if matches:
                nums = re.findall(r"-?\d+", matches[-1])
                if nums:
                    return int(nums[-1])
            # Fallback: last number
            nums = re.findall(r"-?\d+", trace)
            if nums:
                return int(nums[-1])
            return None

        filtered_data = []
        for idx, trace in enumerate(teacher_traces):
            teacher_answer = extract_answer_from_trace(trace)
            row = df.iloc[idx]
            ground_truth_str = str(
                row.get("answer", row.get("expected_answer", row.get("response")))
            )
            import re

            gt_nums = re.findall(r"-?\d+", ground_truth_str)
            ground_truth = int(gt_nums[-1]) if gt_nums else None

            if ground_truth is not None and teacher_answer == ground_truth:
                filtered_data.append(
                    {
                        "problem": row.get("problem", row.get("question", row.get("prompt"))),
                        "answer": ground_truth,
                        "teacher_trace": trace,
                    }
                )

        print(f"  Kept {len(filtered_data)}/{sample_size} samples (teacher matches GT)")

        # Free teacher model memory
        del teacher_model
        gc.collect()
        torch.cuda.empty_cache()
        print("  Teacher memory freed")

    except Exception as e:
        print(f"  Teacher generation failed: {e}")
        print("  Falling back to ground truth only")
        filtered_data = []
        for _, row in df.head(100).iterrows():
            ans_str = str(row.get("answer", row.get("expected_answer", row.get("response"))))
            import re

            gt_nums = re.findall(r"-?\d+", ans_str)
            gt = int(gt_nums[-1]) if gt_nums else 0
            filtered_data.append(
                {
                    "problem": row.get("problem", row.get("question", row.get("prompt"))),
                    "answer": gt,
                    "teacher_trace": "",
                }
            )

    # 6. Load student model (Nemotron-3-Nano-30B-A3B)
    print("\n[6/8] Loading student model...")
    model_id = "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
    model_path = kagglehub.model_download(model_id)
    print(f"  Model path: {model_path}")

    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Ensure we have an offload folder to avoid ValueError
    os.makedirs("/tmp/offload", exist_ok=True)

    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.bfloat16,
        offload_folder="/tmp/offload",
    )

    # Apply LoRA - targeting Attention, Mamba-2, and expert MLPs
    # x_proj and dt_proj are Mamba-2 specific
    target_modules = [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",  # Attention
        "in_proj",
        "out_proj",
        "x_proj",
        "dt_proj",  # Mamba-2
        "w1",
        "w2",
        "w3",  # Experts
    ]

    print(f"  Applying LoRA to target modules: {target_modules}")
    lora_config = LoraConfig(
        r=32,
        lora_alpha=64,  # 2x rank as recommended
        target_modules=target_modules,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    # 7. Prepare dataset for SFT
    print("\n[7/8] Preparing dataset...")

    def format_example(example):
        """Format for SFT with thinking tags."""
        if example["teacher_trace"]:
            # Use <think> tags to preserve reasoning
            text = f"Problem: {example['problem']}\n\n<think>\n{example['teacher_trace']}\n</think>\nAnswer: {example['answer']}"
        else:
            text = f"Problem: {example['problem']}\n\nAnswer: {example['answer']}"
        return {"text": text}

    dataset = Dataset.from_list(filtered_data)
    dataset = dataset.map(format_example)

    # Split for validation
    dataset = dataset.train_test_split(test_size=0.1)
    train_dataset = dataset["train"]
    eval_dataset = dataset["test"]

    print(f"  Train samples: {len(train_dataset)}")
    print(f"  Eval samples: {len(eval_dataset)}")

    # 8. Training
    print("\n[8/8] Starting SFT training...")

    training_args = TrainingArguments(
        output_dir="./nemotron_lora_adapter",
        num_train_epochs=3,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        warmup_steps=10,
        logging_steps=5,
        eval_strategy="steps",
        eval_steps=20,
        save_strategy="steps",
        save_steps=50,
        save_total_limit=2,
        load_best_model_at_end=True,
        bf16=True,
        gradient_checkpointing=True,
        report_to="none",
        remove_unused_columns=False,
    )

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        args=training_args,
        max_seq_length=2048,
    )

    print("\n" + "=" * 60)
    print("TRAINING STARTED")
    print("=" * 60)

    train_result = trainer.train()

    print("\n" + "=" * 60)
    print("TRAINING COMPLETED")
    print("=" * 60)
    print(f"Final loss: {train_result.training_loss:.4f}")

    # Save final adapter
    print("\nSaving trained adapter...")
    trainer.save_model("./nemotron_lora_adapter")
    tokenizer.save_pretrained("./nemotron_lora_adapter")

    # Package submission
    print("\nPackaging submission.zip...")
    subprocess.run(
        "cd nemotron_lora_adapter && zip -r ../submission.zip ./*", shell=True, check=True
    )

    print("\n" + "=" * 60)
    print("SUBMISSION READY: submission.zip")
    print("=" * 60)

except Exception as e:
    print(f"\nERROR: {e}")
    traceback.print_exc()
    sys.exit(1)